# Part 2: Connect Hermes Agent to Nemotron

In Part 1 (`nemotron_deploy_part1.ipynb`) you deployed a Nemotron model on your own system. In this lab you connect Hermes Agent to it.

This lab is based on the NVIDIA playbook [Run Hermes Agent with a Local LLM](https://build.nvidia.com/spark/hermes-agent).

## 1. Start the model server

Part 1 stopped the server at the end. This lab starts it again with a larger context window: `--max-model-len 65536`. Hermes Agent requires a context window of at least 64,000 tokens, and Part 1 used 32,768.

The model is already on disk, so nothing downloads. The next section shows when the server is ready.

In [ ]:
%%bash
docker rm -f vllm-nemotron-nano 2>/dev/null
docker run -d \
  --gpus all --ipc=host --shm-size=16g -p 8000:8000 \
  --name vllm-nemotron-nano \
  -v ~/models/nemotron-3-nano-fp8:/model:ro \
  vllm/vllm-openai:v0.20.0 /model \
  --served-model-name nemotron \
  --max-num-seqs 8 \
  --max-model-len 65536 \
  --gpu-memory-utilization 0.8 \
  --trust-remote-code \
  --enable-prefix-caching \
  --reasoning-parser nemotron_v3 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder


## 2. Wait for the server

The next cell shows a live timer and tells you when the server is ready. It also prints the context window, which should be 65,536 tokens.


In [ ]:
import time, requests

start = time.time()
for _ in range(120):  # up to 10 minutes
    try:
        model = requests.get("http://localhost:8000/v1/models", timeout=3).json()["data"][0]
        print(f"\nServer ready after {time.time() - start:.0f}s. Model: {model['id']}, context window: {model['max_model_len']:,} tokens")
        break
    except Exception:
        print(f"\rLoading the model... {time.time() - start:.0f}s", end="", flush=True)
        time.sleep(5)
else:
    print("\nNot ready after 10 minutes. Check: docker logs --tail 40 vllm-nemotron-nano")


## 3. Install Hermes Agent

Hermes Agent is an open-source agent from Nous Research. This section installs it. The steps differ from the NVIDIA playbook in three ways:

- **The model is Nemotron.** The playbook serves `Qwen3.6-35B-A3B-NVFP4`.
- **The install is non-interactive.** The `--non-interactive` option skips the installer's setup and gateway stages. The setup stage can ask you to sign in to the Nous Portal, and the gateway stage installs a background service. You don't need either here.
- **The context window is 65,536 tokens.** Hermes requires at least 64,000.


In [ ]:
%%bash
curl -fsSL https://raw.githubusercontent.com/NousResearch/hermes-agent/main/scripts/install.sh | bash -s -- --non-interactive 2>&1 | grep -vE '^\s+[A-Za-z-]+:\s+[0-9.]+%'
export PATH="$HOME/.local/bin:$PATH"
which hermes

## 4. Point Hermes at your model

Hermes reads its model settings from its config. These three commands set the provider to `custom` and point it at the vLLM server from section 1, using the served model name `nemotron`.


In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
hermes config set model.provider custom
hermes config set model.base_url http://localhost:8000/v1
hermes config set model.default nemotron


## 5. Check that Hermes uses your model

`hermes -z` sends one prompt and prints the reply. The cell counts the requests vLLM has logged before and after, so you can see the request reached your server.


In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
count() { docker logs vllm-nemotron-nano 2>&1 | grep -c "POST /v1/chat/completions"; }

before=$(count)
hermes -z "Reply exactly HERMES_OK"
after=$(count)
echo "vLLM requests before: $before, after: $after"

## 6. Run a tool call

Hermes can run shell commands. This cell creates a folder with three files, then asks Hermes to list them. To answer, the model has to call a tool, and the tool result goes back to the model, so vLLM should log at least 2 new requests.

Hermes' terminal backend is `local`, so the command runs directly on this machine.

In testing, the model once printed the tool call as text instead of running it. If you see a block with `calls` and no file names, run the cell again.

In [ ]:
%%bash
export PATH="$HOME/.local/bin:$PATH"
count() { docker logs vllm-nemotron-nano 2>&1 | grep -c "POST /v1/chat/completions"; }

mkdir -p /tmp/hermes_demo && cd /tmp/hermes_demo
touch alpha.txt beta.txt gamma.txt

before=$(count)
hermes -z "Use the terminal to list the files in the current directory and tell me their names."
after=$(count)
echo
echo "New vLLM requests: $((after - before))"


## Next

Leave the model server running. In Part 3 you use it with OpenShell.
